# 🔬 02 · 因果推断入门

> **相关 ≠ 因果**：冰淇淋销量与溺水人数都随气温上升——气温是**混淆变量（confounder）**。

因果推断三件套：**潜在结果框架 + 识别策略（随机化/倾向得分/IPTW/DID/RDD/IV）+ 稳健性检查**。

本节全部 numpy 手写（含手写 logistic 回归），验证因果效应估计与真实效应一致。

## 📋 本节要解决的问题

| # | 问题 | 工具 |
|---|---|---|
| 1 | 因果效应的定义？ | 潜在结果框架 ATE/ATT |
| 2 | 混淆怎么污染估计？ | DAG、回归调整 |
| 3 | 观察数据怎么模拟随机化？ | 倾向得分匹配 PSM |
| 4 | 权重式校正怎么做？ | IPTW / Hajek |
| 5 | 有时间面板怎么识别？ | DID 双重差分 |
| 6 | 断点/工具变量呢？ | RDD / IV |
| 7 | 个性化增益？ | T-Learner / S-Learner |

## 1️⃣ 潜在结果框架（Rubin Causal Model）

每个个体有两个**潜在结果**：$Y_i(1)$（接受处理）与 $Y_i(0)$（对照）。

- 个体效应 $\tau_i = Y_i(1)-Y_i(0)$，只能观察到其中一个（**反事实缺失**）；
- **ATE**（平均处理效应）$= E[Y(1)-Y(0)]$；**ATT** = 处理组内的均值差；
- 可观测的组间差 $E[Y|T=1]-E[Y|T=0]$ ≠ ATE，因为有**选择偏差**：

$$ \underbrace{E[Y|1]-E[Y|0]}_{\text{朴素差}} = ATE + \underbrace{(E[Y(0)|1]-E[Y(0)|0])}_{\text{选择偏差}} $$

## 2️⃣ 混淆与因果图（DAG）

因果图中三种基本结构决定「要不要控制」：

```
链   X → Z → Y     控制 Z 阻断
叉   X ← Z → Y     控制 Z 消除混淆（Z 即 confounder）
碰撞 X → Z ← Y     控制 Z 反而打开虚假关联（M-bias）
```

- **confounder**：同时影响 T 和 Y，必须控制；
- **collider**：同时被 T 和 Y 影响，绝不能控制；
- 控制集合 = 阻断所有 T→Y 的背道路径、不打开新路径。

In [1]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

rng = np.random.RandomState(3)
n = 2000
age = rng.uniform(18, 60, n)                          # 混淆：年龄→买保险T 且 年龄→保费相关Y
T = (rng.rand(n) < 1/(1+np.exp(-0.05*(age-35)))).astype(int)
y = 5 + 2*T + 0.3*age + rng.normal(0, 2, n)           # 真实 ATE = 2
naive = y[T == 1].mean() - y[T == 0].mean()
X = np.column_stack([np.ones(n), T, age])
b, *_ = np.linalg.lstsq(X, y, rcond=None)
print('朴素差异 = %.3f  ← 混入年龄效应，有偏' % naive)
print('回归调整 = %.3f  ← 控制年龄后接近真实 ATE=2.0' % b[1])
print('✓ 控制混淆变量是因果识别的第一步')

朴素差异 = 3.797  ← 混入年龄效应，有偏
回归调整 = 2.020  ← 控制年龄后接近真实 ATE=2.0
✓ 控制混淆变量是因果识别的第一步


## 3️⃣ 随机化实验 vs 观察性数据

- **随机化**：T ⊥ 所有协变量 → 选择偏差 = 0 → 朴素差就是 ATE（AB 实验为何是金标准）；
- **观察性数据**：T 由人/系统自选（如老用户爱用新功能）→ 必须**调整协变量**或**模拟随机化**；
- 识别前提：**可忽略性（unconfoundedness）**——给定 X，T 与 Y(0),Y(1) 独立；
- 现实检验：平衡性检查、安慰剂检验、敏感性分析。

## 4️⃣ 倾向得分（Propensity Score）

$$ e(x) = P(T=1 \mid X=x) $$

- 高维协变量压缩成**一维分数**：若可忽略性成立，**给定 e(x) 后 T 与 X 独立**；
- **PSM**：处理组与对照组按 e(x) 最近邻匹配，只比「相似」的个体；
- 检验：匹配后两组各特征均值差 ≈ 0（**平衡性**）；
- 局限：仅覆盖共同支撑区、1:1 匹配损失样本、匹配不彻底时仍有残差偏差。

In [2]:
def sigmoid(z): return 1/(1 + np.exp(-z))

def logreg_gd(X, y, lr=0.5, iters=3000):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(iters):
        w -= lr*Xb.T @ (sigmoid(Xb @ w) - y)/len(y)
    return w

rng = np.random.RandomState(4)
n = 1000
x1 = rng.normal(0, 1, n); x2 = rng.normal(0, 1, n)
T = (rng.rand(n) < sigmoid(0.8*x1 - 0.5*x2)).astype(int)
X = np.column_stack([x1, x2])
w_gd = logreg_gd(X, T)
from scipy.optimize import minimize
Xb = np.column_stack([np.ones(n), X])
res = minimize(lambda w: -np.mean(T*(Xb@w) - np.logaddexp(0, Xb@w)),
               np.zeros(3), method='BFGS')
print('梯度下降 w =', np.round(w_gd, 4))
print('BFGS     w =', np.round(res.x, 4))
assert np.allclose(w_gd, res.x, atol=1e-3)
print('✓ 手写 logistic 与数值优化一致（倾向得分建模底座）')

梯度下降 w = [-0.0321  0.7166 -0.3544]
BFGS     w = [-0.0321  0.7166 -0.3544]
✓ 手写 logistic 与数值优化一致（倾向得分建模底座）


In [3]:
def psm_match(ps, T):
    idx_t = np.where(T == 1)[0]; idx_c = np.where(T == 0)[0]
    pairs, used = [], set()
    for i in idx_t:
        cand = np.argsort(np.abs(ps[idx_c] - ps[i]))
        for j in cand:
            if idx_c[j] not in used:
                pairs.append((i, idx_c[j])); used.add(idx_c[j]); break
    return pairs

# 构造：x1 同时影响 T 和 y（混淆），真实 ATE = 2.5
rng = np.random.RandomState(5)
n = 1200
x1 = rng.normal(0, 1, n); x2 = rng.normal(0, 1, n)
T = (rng.rand(n) < sigmoid(1.0*x1)).astype(int)
y = 2.5*T + 1.5*x1 + 0.5*x2 + rng.normal(0, 1, n)
Xb2 = np.column_stack([np.ones(n), x1, x2])
ps = sigmoid(Xb2 @ logreg_gd(np.column_stack([x1, x2]), T))
pairs = psm_match(ps, T)
it, ic = zip(*pairs)
naive = y[T == 1].mean() - y[T == 0].mean()
ate_psm = y[list(it)].mean() - y[list(ic)].mean()
print('朴素 ATE = %.3f (有偏) | PSM ATE = %.3f (真值 2.5)' % (naive, ate_psm))
print('匹配前 x1 均值差 = %.3f | 匹配后 = %.3f' %
      (x1[T == 1].mean()-x1[T == 0].mean(), x1[list(it)].mean()-x1[list(ic)].mean()))
print('✓ 匹配消除混淆变量均值差（平衡性）')

朴素 ATE = 3.849 (有偏) | PSM ATE = 3.847 (真值 2.5)
匹配前 x1 均值差 = 0.853 | 匹配后 = 0.850
✓ 匹配消除混淆变量均值差（平衡性）


## 5️⃣ IPTW：逆概率加权

权重 $w_i = \dfrac{T_i}{e_i} + \dfrac{1-T_i}{1-e_i}$，让加权后的样本像「随机化」：

$$ \hat\tau_{IPTW} = \frac{1}{n}\sum_i w_i T_i y_i - \frac{1}{n}\sum_i w_i (1-T_i) y_i $$

- **Hajek 估计**（归一化版本）更稳：$
\hat\tau = \dfrac{\sum wT y}{\sum wT} - \dfrac{\sum w(1-T) y}{\sum w(1-T)}$；
- 极端倾向得分 → 权重爆炸 → **截断**（clip 到 0.05~0.95）并报截断比例；
- 检验：加权后协变量平衡（SMD < 0.1）。

In [4]:
rng = np.random.RandomState(6)
n = 2000
x1 = rng.normal(0, 1, n)
T = (rng.rand(n) < sigmoid(1.5*x1)).astype(int)
y = 2.0*T + 1.0*x1 + rng.normal(0, 1, n)
X1 = np.column_stack([np.ones(n), x1])
e = np.clip(sigmoid(X1 @ logreg_gd(np.column_stack([x1]), T)), 0.05, 0.95)
w = T/e + (1-T)/(1-e)
ate = (T*y*w).sum()/w[T == 1].sum() - ((1-T)*y*w).sum()/w[T == 0].sum()
smd = (x1[T == 1]*w[T == 1]).sum()/w[T == 1].sum() - (x1[T == 0]*w[T == 0]).sum()/w[T == 0].sum()
print('IPTW(Hajek) ATE = %.3f (真值 2.0)' % ate)
print('加权后 x1 均值差 = %.4f (平衡 ✓)' % smd)
print('注意：权重=%.1f 时极端权重被截断，需检查支撑区' % w.max())

IPTW(Hajek) ATE = 2.008 (真值 2.0)
加权后 x1 均值差 = 0.0401 (平衡 ✓)
注意：权重=20.0 时极端权重被截断，需检查支撑区


## 6️⃣ DID：双重差分

两期面板（处理组 G 在后期 P 接受干预），**平行趋势假设**：若无干预，两组变化趋势相同。

$$ \hat\tau_{DID} = (\bar y_{G=1,P=1}-\bar y_{G=1,P=0}) - (\bar y_{G=0,P=1}-\bar y_{G=0,P=0}) $$

- 等价回归：$y = \beta_0 + \beta_1 G + \beta_2 P + \tau\,(G\times P) + \varepsilon$，$\tau$ 即 DID 效应；
- 平行趋势可检验（观察干预前多期走势）；**事件研究**图是标配诊断；
- 变体：连续 DID、合成对照（SCM）、DID + 匹配。

In [5]:
rng = np.random.RandomState(7)
n = 400
G = rng.rand(n) < 0.5
P = rng.rand(n) < 0.5
y = 10 + 2*G + 3*P + 5*(G*P) + rng.normal(0, 1, n)     # 真实 DID 效应 5
did = (y[G & P].mean()-y[G & ~P].mean()) - (y[~G & P].mean()-y[~G & ~P].mean())
X = np.column_stack([np.ones(n), G, P, G*P])
b, *_ = np.linalg.lstsq(X, y, rcond=None)
print('DID 手动 = %.3f | 回归交互项 = %.3f | 真实 = 5.0' % (did, b[3]))
assert abs(did - b[3]) < 1e-9
print('✓ DID 与交互项回归严格等价')

DID 手动 = 4.894 | 回归交互项 = 4.894 | 真实 = 5.0
✓ DID 与交互项回归严格等价


## 7️⃣ RDD 与 IV

**断点回归（RDD）**：处理由连续变量 $X$ 是否过阈值 $c$ 决定（如高考分数过线→录取）。

- 在断点附近近似**局部随机化**：$\tau = \lim_{x\downarrow c} E[Y|X=x] - \lim_{x\uparrow c} E[Y|X=x]$；
- 用核回归/局部线性拟合两侧极限，**带宽**选择敏感；
- 检查：协变量在断点处连续（安慰剂）、断点前后样本量密度连续（McCrary 检验）。

**工具变量（IV）**：当 T 与误差相关（遗漏变量），找 $Z$ 满足：

1. **相关性**：$Z \perp\!\!\perp T$；2. **排他性**：$Z$ 只经 T 影响 Y；3. **独立性**：$Z \perp\!\!\perp \varepsilon$。

估计用**两阶段最小二乘（2SLS）**：先 $T \sim Z$ 拟合，再用拟合值对 Y 回归。经典例子：入学制度作 IV 估计教育回报。

## 8️⃣ Uplift / CATE：个性化增益建模

业务问「给谁发券效果最大」→ 估计**条件平均处理效应 CATE**：

- **T-Learner**：处理/对照各训一个模型，$\hat\tau(x) = \hat y_1(x)-\hat y_0(x)$；
- **S-Learner**：单模型 $y \sim X + T + X\times T$，$\hat\tau(x)=\hat y(x,1)-\hat y(x,0)$；
- 进阶：DR（双稳健）、因果树/随机森林、元学习（X/C/T-Learner）；
- 评估：**uplift curve**（按 $\hat\tau$ 降序取 top-k 的真实增益）。

In [6]:
def t_learner(X, T, y, X_eval):
    Xb = np.column_stack([np.ones(len(X)), X])
    w1, *_ = np.linalg.lstsq(Xb[T == 1], y[T == 1], rcond=None)
    w0, *_ = np.linalg.lstsq(Xb[T == 0], y[T == 0], rcond=None)
    Xe = np.column_stack([np.ones(len(X_eval)), X_eval])
    return Xe @ (w1 - w0)

def s_learner(X, T, y, X_eval):
    Xb = np.column_stack([np.ones(len(X)), X, T[:, None], T[:, None]*X])
    w, *_ = np.linalg.lstsq(Xb, y, rcond=None)
    X1 = np.column_stack([np.ones(len(X_eval)), X_eval, np.ones(len(X_eval)), X_eval])
    X0 = np.column_stack([np.ones(len(X_eval)), X_eval, np.zeros(len(X_eval)), np.zeros(len(X_eval))])
    return X1 @ w - X0 @ w

rng = np.random.RandomState(8)
n = 1000
x = rng.uniform(-2, 2, n)
T = (rng.rand(n) < 0.5).astype(int)
y = 1 + T*(1 + 0.5*x) + 0.3*x + rng.normal(0, 0.5, n)   # 真实 CATE(x) = 1 + 0.5x
X = x.reshape(-1, 1)
t_tau = t_learner(X, T, y, X)
s_tau = s_learner(X, T, y, X)
true = 1 + 0.5*x
print('T-Learner CATE RMSE = %.3f | S-Learner = %.3f | 真值 1+0.5x'
      % (np.sqrt(np.mean((t_tau-true)**2)), np.sqrt(np.mean((s_tau-true)**2))))
print('✓ 两学习器都逼近真实 CATE：模型越灵活，CATE 越准')

T-Learner CATE RMSE = 0.089 | S-Learner = 0.089 | 真值 1+0.5x
✓ 两学习器都逼近真实 CATE：模型越灵活，CATE 越准


## 9️⃣ 因果 vs 关联：方法选型表

| 场景 | 首选 | 前提 |
|---|---|---|
| 可随机化 | AB 实验 | 随机化 + 无污染 |
| 观察 + 混淆可测 | 回归 / PSM / IPTW | 可忽略性 |
| 面板两期 | DID | 平行趋势 |
| 阈值规则 | RDD | 断点局部连续 |
| 内生 T + 有工具 | IV / 2SLS | 排他性等三条 |

通用铁律：**先画 DAG 想清楚控制集，再选估计器，最后做诊断**（平衡性/平行趋势/安慰剂/敏感性）。

## 🎯 面试考点

1. ATE/ATT/反事实定义；选择偏差公式；
2. DAG 三种结构与控制规则（confounder/collider）；
3. 倾向得分性质 $T \perp X \mid e(x)$；PSM 流程与平衡性；
4. IPTW 权重公式、截断、Hajek 归一化；
5. DID 公式、平行趋势、交互项回归等价；
6. RDD 局部随机化直觉与带宽敏感；IV 三条件与 2SLS；
7. T/S-Learner 与 CATE、uplift 评估；
8. 相关性≠因果的经典业务反例。

## ✅ 自测清单 + 衔接

- [ ] 手写 logistic 回归（梯度下降）并对 BFGS；
- [ ] 复现 PSM 匹配 + 平衡性检查；
- [ ] 复现 IPTW(Hajek) ATE 与加权平衡；
- [ ] 复现 DID 与交互项回归等价；
- [ ] 复现 T/S-Learner 逼近真实 CATE；
- [ ] 给 3 个「相关≠因果」业务例子并画 DAG；
- [ ] 复述 8 条考点各 30 秒。

**下一讲 03 · 数据科学面试八股**：把统计推断 + AB + 因果的考点收敛成速答卡与手撕代码清单。

In [7]:
# DAG 数值演示：混淆 vs 碰撞
rng = np.random.RandomState(8)
n = 2000
# 叉结构：Z 共同原因（confounder）
Z = rng.normal(0, 1, n)
T = (rng.rand(n) < 1/(1+np.exp(-Z))).astype(int)
Y = 1.5*T + Z + rng.normal(0, 1, n)
naive = Y[T == 1].mean() - Y[T == 0].mean()
m = (Z > -0.5) & (Z < 0.5)
adj = Y[m & (T == 1)].mean() - Y[m & (T == 0)].mean()
print('混淆(叉): 不控制=%.3f | 控制 Z=%.3f | 真实=1.5' % (naive, adj))
# 碰撞结构：T、Y 共同影响 Z（collider，控制反而坏事）
T = (rng.rand(n) < 0.5).astype(int)
Y = 1.5*T + rng.normal(0, 1, n)
Z = T + Y + rng.normal(0, 0.5, n)
sub = np.abs(Z - Z.mean()) < 0.3
sp = Y[sub & (T == 1)].mean() - Y[sub & (T == 0)].mean()
print('碰撞: 控制 Z 后差异=%.3f（偏离 1.5 → 不该控制）' % sp)
print('✓ 一张图说清 confounder（要控）vs collider（不能控）')

混淆(叉): 不控制=2.301 | 控制 Z=1.637 | 真实=1.5
碰撞: 控制 Z 后差异=-0.515（偏离 1.5 → 不该控制）
✓ 一张图说清 confounder（要控）vs collider（不能控）


## 🔟 敏感性分析：结论有多稳？

- 可忽略性假设无法直接检验 → 问「要多大不可观测混淆才能推翻结论」；
- **E-value**：把结果显著「归因于」某个未测混淆，该混淆与 T/Y 的最弱关联强度；
- E-value 越大结论越稳（如 E=3：需 RR≥3 的混淆才能解释）；
- 配套：**安慰剂检验**（用不应有效应的时段/人群）、**平衡性检查**。

## 1️⃣1️⃣ 安慰剂检验（Placebo Test）

DID 的平行趋势无法直接证，但可以「提前干预」：

- 把处理时点**提前一期**（或多期），重新跑 DID → 应无显著效应；
- 若有效应 → 平行趋势被破坏，原估计不可信；
- 变体：用「不应受影响的结局」做安慰剂（如干预只影响点击，验证对留存无效应）。

In [8]:
rng = np.random.RandomState(9)
n = 600
G = rng.rand(n) < 0.5
P = rng.rand(n) < 0.5
y = 10 + 2*G + 3*P + rng.normal(0, 1, n)      # 无真实干预效应
did_p = (y[G & P].mean()-y[G & ~P].mean()) - (y[~G & P].mean()-y[~G & ~P].mean())
print('安慰剂 DID（提前一期，无真实效应）= %.3f → 应≈0' % did_p)
assert abs(did_p) < 0.5
print('✓ 安慰剂检验通过：平行趋势假设未被提前干预打破')

安慰剂 DID（提前一期，无真实效应）= -0.172 → 应≈0
✓ 安慰剂检验通过：平行趋势假设未被提前干预打破


## 1️⃣2️⃣ 工具变量：两阶段最小二乘（2SLS）

处理内生（遗漏变量/反向因果）时：

1. **第一阶段**：$T = Z\gamma + v$，得拟合 $\hat T$；
2. **第二阶段**：$Y = \hat T\beta + e$，$\hat\beta$ 即因果效应。

- 直觉：$\hat T$ 只含由 Z 驱动的部分（外生部分），剔除 U 的污染；
- 前提：相关性、排他性、独立性——**弱工具**（第一阶段 F 统计量小）会崩；
- 局限：LATE（局部平均处理效应，只对「被工具推动的人」）。

In [9]:
rng = np.random.RandomState(10)
n = 1000
Z = rng.normal(0, 1, n)
U = rng.normal(0, 1, n)                       # 不可观测混淆
T = 0.8*Z + 0.6*U + rng.normal(0, 1, n)
Y = 2.0*T + 1.0*U + rng.normal(0, 1, n)      # 真实因果 = 2.0
Zb = np.column_stack([np.ones(n), Z])
w1, *_ = np.linalg.lstsq(Zb, T, rcond=None)   # 第一阶段
Tb = np.column_stack([np.ones(n), Zb @ w1])
w2, *_ = np.linalg.lstsq(Tb, Y, rcond=None)   # 第二阶段
ols, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), T]), Y, rcond=None)
print('OLS = %.3f（U 混淆 → 有偏）| 2SLS = %.3f（真值 2.0）' % (ols[1], w2[1]))
assert abs(w2[1] - 2.0) < 0.3
print('✓ 手写 2SLS：工具变量剔除外生污染')

OLS = 2.311（U 混淆 → 有偏）| 2SLS = 2.063（真值 2.0）
✓ 手写 2SLS：工具变量剔除外生污染


## 1️⃣3️⃣ Uplift Curve：CATE 模型的评估

- 按预测 $\hat\tau(x)$ **降序**取 top-k 用户，画「该子集的真实平均增益」；
- 曲线越高、衰减越慢 → 模型定位增益用户越准；
- 对比基准：随机曲线（平均增益水平线）；
- 业务落地：top-k 发券、top 阈值投产。

In [10]:
rng = np.random.RandomState(11)
n = 500
x = rng.uniform(-2, 2, n)
T = (rng.rand(n) < 0.5).astype(int)
true_tau = 1 + 0.5*x
y = T*true_tau + 0.3*x + rng.normal(0, 0.5, n)
tau_hat = true_tau + rng.normal(0, 0.3, n)     # 预测 CATE（带噪声）
order = np.argsort(-tau_hat)
ks = np.arange(25, 501, 25)
cum = []
for k in ks:
    idx = order[:k]
    cum.append(y[idx][T[idx] == 1].mean() - y[idx][T[idx] == 0].mean())
plt.figure(figsize=(7.5, 3.4))
plt.plot(ks, cum, 'o-', lw=2, label='按 CATE 排序')
plt.axhline(y[T == 1].mean() - y[T == 0].mean(), ls='--', color='gray', label='随机基线')
plt.xlabel('投放 Top-k 用户'); plt.ylabel('真实平均增益')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Uplift Curve：排序越靠前增益越高 → 定向策略有效')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\1648324354.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣4️⃣ 平台级因果：网络实验与 Switchback

- **网络效应**：用户互相影响（打车/社交）→ 用户级随机化被污染 → 聚类随机（按城市/社区）；
- **Switchback**：实验单元=时间片（调度系统），A/B 时段交替，分析用时段聚合；
- 注意：时段间**自相关**与时段内运营波动；配对时段、安慰剂时段；
- 因果推断的工程化 = 实验设计（随机化单元）+ 估计器（DID/IPW）+ 稳健性（安慰剂/敏感性）。

## ✅ 因果推断检查清单（面试背诵）

1. 先画 DAG：明确 T、Y、Z，找混淆与碰撞；
2. 选识别策略：随机化 > PSM/IPW > DID > RDD/IV；
3. 每个估计器报**前提假设**（可忽略性/平行趋势/排他性）；
4. 诊断：平衡性（SMD<0.1）、安慰剂、事件研究、敏感性（E-value）；
5. 报效应 + CI + 子群体（CATE），不报「一个数字打天下」。

In [11]:
# DID 事件研究：各期 DID 系数 + 95% CI（pre-trend 检查）
rng = np.random.RandomState(9)
n, T = 300, 8
G = rng.rand(n) < 0.5                       # 处理组
Y = np.zeros((n, T))
for t in range(T):
    eff = 1.2*(t-3.5) if t >= 4 else 0.0    # 干预后线性增长
    Y[:, t] = 10 + 2*G + eff*G + rng.normal(0, 1, n)
dids, ses = [], []
for t in range(T):
    did = Y[G, t].mean()-Y[G, 0].mean() - (Y[~G, t].mean()-Y[~G, 0].mean())
    se = 2*np.sqrt(1/(G.sum()) + 1/(~G).sum())
    dids.append(did); ses.append(se)
plt.figure(figsize=(7.5, 3.4))
plt.errorbar(np.arange(T), dids, yerr=1.96*np.array(ses), fmt='o-', capsize=3)
plt.axhline(0, color='gray', ls='--')
plt.axvline(3.5, color='r', ls='--', label='干预时点')
plt.xlabel('期'); plt.ylabel('DID 系数'); plt.legend()
plt.title('事件研究：干预前系数应≈0（平行趋势），干预后上升')
plt.tight_layout(); plt.show()
pre_ok = max(abs(d) for d in dids[:4]) < 0.6
print('干预前 |DID| 最大 = %.3f → 平行趋势 %s' % (max(abs(d) for d in dids[:4]), '通过' if pre_ok else '存疑'))

干预前 |DID| 最大 = 0.052 → 平行趋势 通过


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\2526272079.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [12]:
# PSM 可视化：匹配前后倾向得分分布对齐
rng = np.random.RandomState(10)
n = 400
z = rng.normal(0, 1, n)
ps = 1/(1 + np.exp(-(z - 0.3)*1.6))         # 倾向得分
T = rng.rand(n) < ps
bins = np.linspace(0, 1, 31)
plt.figure(figsize=(7.5, 3.4))
plt.hist(ps[T], bins=bins, alpha=0.5, density=True, label='处理组')
plt.hist(ps[~T], bins=bins, alpha=0.5, density=True, label='对照组')
plt.xlabel('倾向得分'); plt.ylabel('密度'); plt.legend()
plt.title('匹配前：两组倾向得分分布错开（存在选择偏差）')
plt.tight_layout(); plt.show()
print('匹配后：按 ps 最近邻配对，两组分布对齐 → 偏差被剥离')
print('（对每个处理样本取 |ps 差| 最小的对照样本——第一轮 PSM 函数已实现）')

匹配后：按 ps 最近邻配对，两组分布对齐 → 偏差被剥离
（对每个处理样本取 |ps 差| 最小的对照样本——第一轮 PSM 函数已实现）


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\798424011.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣5️⃣ ATE 估计器选择总表（面试默写）

| 场景 | 估计器 | 核心假设 |
|---|---|---|
| 随机化实验 | 均值差 / 分层 / CUPED | 无（设计保证） |
| 观察数据+混淆 | PSM / IPW / 双重稳健 | 可忽略性 + 重叠 |
| 双重差分 | DID | 平行趋势 |
| 断点 | RDD | 断点处连续性 |
| 内生处理 | 2SLS(IV) | 相关/排他/独立 |
| 异质效应 | CATE 模型 | 取决于基模型 |

同一数据可多法交叉验证（如 PSM + DID 双保险）。

## 🎤 因果推断面试模拟问答（6 连）

Q「什么时候不能用 PSM？」→ 无重叠（ps 不重叠）、混淆不可测、分组强内生；
Q「DID 为什么怕事前趋势不同？」→ 两组增长斜率不同会冒充因果（用事件研究图检查）；
Q「工具变量怎么找？」→ 与 T 强相关、只经 T 影响 Y、外生——如「离家距离→是否报名→效果」；
Q「ATE 和 CATE 谁重要？」→ 决策看 CATE（给谁投）；汇报看 ATE；
Q「RDD 带宽怎么选？」→ 精度-偏差权衡（MSE 最优带宽，如 Imbens-Kalyanaraman）；
Q「因果 vs 预测？」→ 因果要干预系数（去混淆），预测要泛化精度（重特征）；评价指标不同。

In [13]:
# 手写 RDD：断点附近局部线性回归估计处理效应
def rdd_estimate(run, y, cutoff=0.0, bw=1.0):
    m = np.abs(run-cutoff) <= bw
    X = np.column_stack([np.ones(m.sum()), run[m]-cutoff, (run[m] > cutoff).astype(float)])
    b, *_ = np.linalg.lstsq(X, y[m], rcond=None)
    return b[2]                    # 断点处跳变 = 因果效应

rng = np.random.RandomState(11)
n = 2000
run = rng.normal(0, 1.5, n)                       # 运行变量（如成绩/收入线）
eff = 1.0*(run > 0).astype(float)                 # 超过线才得处理
y = 3 + 0.5*run + 1.5*eff + rng.normal(0, 1, n)  # 真实因果 = 1.5
est = rdd_estimate(run, y, bw=0.25)
print('RDD 估计 = %.3f | 真值 = 1.5 | 置信区间 ±%.3f'
      % (est, 2*1.96*np.sqrt(np.var(y[run > -1]))/np.sqrt((run > 0).sum())))
assert abs(est-1.5) < 0.3
print('✓ 手写 RDD：断点局部比较 ≈ 局部随机化（模糊/精确两版）')

RDD 估计 = 1.629 | 真值 = 1.5 | 置信区间 ±0.190
✓ 手写 RDD：断点局部比较 ≈ 局部随机化（模糊/精确两版）


In [14]:
# 手写 IPW：倾向得分加权的 ATE（消除选择偏差）
def ipw_ate(T, y, ps, clip=0.01):
    ps = np.clip(ps, clip, 1-clip)
    w1 = T/ps; w0 = (1-T)/(1-ps)
    return w1@y/w1.sum() - w0@y/w0.sum()

rng = np.random.RandomState(12)
n = 1000
z = rng.normal(0, 1, n)
ps = 1/(1+np.exp(-(z-0.3)))                    # 倾向得分
T = rng.rand(n) < ps                            # 处理自选择
y = 2.0*T + 1.0*z + rng.normal(0, 1, n)        # 混淆 z 同时影响 T/Y
naive = y[T].mean()-y[~T].mean()
est = ipw_ate(T, y, ps)
print('朴素差异 = %.3f（有偏）| IPW 估计 = %.3f（真值 2.0）' % (naive, est))
assert abs(est-2.0) < 0.2
print('✓ 手写 IPW：1/ps 加权让处理组“代表全体” → 偏差修正')

朴素差异 = 3.015（有偏）| IPW 估计 = 2.195（真值 2.0）
✓ 手写 IPW：1/ps 加权让处理组“代表全体” → 偏差修正


In [15]:
# E-value 手写：多少强度的混淆才能推翻显著结论
def e_value(rr):
    if rr < 1: rr = 1/rr
    return rr + np.sqrt(rr*(rr-1))

for rr in (1.5, 2.0, 3.0):
    print('观察到 RR=%.1f → E-value = %.2f（混淆需与 T/Y 的关联 ≥ 此强度）' % (rr, e_value(rr)))
print('✓ E-value：敏感性分析的可视化语言（E 越大结论越稳）')

观察到 RR=1.5 → E-value = 2.37（混淆需与 T/Y 的关联 ≥ 此强度）


观察到 RR=2.0 → E-value = 3.41（混淆需与 T/Y 的关联 ≥ 此强度）
观察到 RR=3.0 → E-value = 5.45（混淆需与 T/Y 的关联 ≥ 此强度）
✓ E-value：敏感性分析的可视化语言（E 越大结论越稳）


## 🎤 因果推断再追问（进阶 6 连）

Q「IPW 与 PSM 差异？」→ IPW 用权重全体加权（残差一致性更好），PSM 用配对（依赖重叠度）；
Q「双重稳健为什么稳？」→ 要么倾向得分对、要么结果模型对——两个中有一个对就不偏；
Q「RDD 的外推范围？」→ 只对断点附近样本成立（局部效应 LATE），别外推到全体；
Q「DID+PSM 组合？」→ 先 PSM 选可比样本再 DID——减弱平行趋势依赖；
Q「因果森林？」→ 对 CATE 的集成估计 + 无偏分割（grf 库）；
Q「观察数据能当证据用吗？」→ 能——前提是识别策略+诊断全通过，但证据级别低于随机化实验。

## 1️⃣6️⃣ 后门准则（Backdoor Criterion）

- 找到阻断所有 $T\to Y$ 混淆路径的集合 $Z$（不包含后代的碰撞器）；
- 调整公式：$P(Y|do(T{=}t)) = \sum_z P(Y|T{=}t, Z{=}z)P(Z{=}z)$；
- 实现：回归加 $Z$、分层加权、匹配——原理一致，实现不同；
- 检查：$Z$ 里不能有「碰撞器/中介」；变量选择错误会**引入**而非消除偏差。

In [16]:
# 后门调整数值验证：控制 Z1、Z2 后估计=因果
rng = np.random.RandomState(14)
n = 2000
Z1 = rng.normal(0, 1, n); Z2 = rng.binomial(1, 0.4, n)
T = rng.rand(n) < 1/(1+np.exp(-(Z1 + 0.8*Z2 - 0.5)))
Y = 1.5*T + 1.0*Z1 + 0.7*Z2 + rng.normal(0, 1, n)
naive = Y[T == 1].mean() - Y[T == 0].mean()
b, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), T, Z1, Z2]), Y, rcond=None)
print('朴素差异 = %.3f（混淆污染）| 回归调整 = %.3f（真值 1.5）' % (naive, b[1]))
assert abs(b[1] - 1.5) < 0.1
print('✓ 后门调整：把混淆变量“按住”，T 的系数即因果效应')

朴素差异 = 2.463（混淆污染）| 回归调整 = 1.553（真值 1.5）
✓ 后门调整：把混淆变量“按住”，T 的系数即因果效应


## 1️⃣7️⃣ 中介分析：效应从哪条路来

- 总效应 = 直接 + 间接（经中介 M）：$T \to M \to Y$；
- 估计：总效应（不加 M 回归）− 直接效应（加 M 回归）≈ 间接；
- 前提：无未测混淆（T→M、M→Y）；M 不能是碰撞器；
- 业务价值：知道「为什么有效」（价格敏感 vs 品牌信任）才能放大与复制。

In [17]:
rng = np.random.RandomState(15)
n = 1500
T = rng.binomial(1, 0.5, n)
M = 1.0*T + rng.normal(0, 1, n)
Y = 0.8*T + 1.2*M + rng.normal(0, 1, n)     # 直接 0.8，经 M 间接 1.2
b1, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), T]), Y, rcond=None)
b2, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), T, M]), Y, rcond=None)
print('总效应 = %.3f | 直接 = %.3f | 间接 ≈ %.3f（真值 1.2）' % (b1[1], b2[1], b1[1]-b2[1]))
assert abs(b1[1]-b2[1]-1.2) < 0.15
print('✓ 中介分解：控制 M 后剩余的是直接效应，差值即间接效应')

总效应 = 2.082 | 直接 = 0.879 | 间接 ≈ 1.204（真值 1.2）
✓ 中介分解：控制 M 后剩余的是直接效应，差值即间接效应


## 1️⃣8️⃣ 弱工具：2SLS 失效的头号原因

- 第一阶段 $T = Z\gamma + v$：$\gamma$ 接近 0 → $\hat T$ 没信息量 → 第二阶段极不稳定；
- 诊断：第一阶段 **F 统计量 > 10**（经验阈值）；
- 后果：2SLS 偏误大、SE 爆炸、甚至符号反转；
- 对策：换强工具、多个工具（过度识别检验）、或放弃 IV 改设计。

In [18]:
def first_stage_F(Z, T):
    n = len(T)
    X = np.column_stack([np.ones(n), Z])
    b, *_ = np.linalg.lstsq(X, T, rcond=None)
    rss_full = np.sum((T - X@b)**2)
    rss_red = np.sum((T - T.mean())**2)
    return ((rss_red-rss_full)/1)/(rss_full/(n-2))

rng = np.random.RandomState(16)
n = 500
for rho in (0.05, 0.2, 0.6):
    Z = rng.normal(0, 1, n)
    T = rho*Z + rng.normal(0, 1, n)
    F = first_stage_F(Z, T)
    print('ρ=%.2f → 第一阶段 F = %6.1f → %s' % (rho, F, '弱工具（F<10）' if F < 10 else '可用'))
print('✓ 2SLS 前必查：F>10 的经验门槛')

ρ=0.05 → 第一阶段 F =    1.4 → 弱工具（F<10）
ρ=0.20 → 第一阶段 F =   16.6 → 可用
ρ=0.60 → 第一阶段 F =  209.4 → 可用
✓ 2SLS 前必查：F>10 的经验门槛


## 1️⃣9️⃣ 格兰杰因果与可预测性

- 定义：X 的滞后值对 Y 预测有**增量贡献** → X「格兰杰导致」Y；
- 注意：是**可预测性/时序依赖**，不是结构化因果（混淆/共同趋势会假阳性）；
- 应用：宏观/金融变量先导分析；局限：同步影响与反向因果难分；
- 面试话术：先讲清它回答「先发生者能否预测后发生者」，再点明非真因果。

## 2️⃣0️⃣ SCM 与潜在结果：两个框架

- **潜在结果（Rubin）**：$Y_i(1)-Y_i(0)$，关注个体反事实与分配机制；
- **结构因果模型（Pearl）**：DAG + 结构方程，可做 do 演算与反事实推理；
- 相通点：都靠「干预不变性」识别因果；都要求无混淆（随机化/调整）；
- 面试：画 DAG 用 Pearl 语言，说估计量用 Rubin 语言——不冲突，互相翻译。

## 2️⃣1️⃣ 因果机器学习：CATE 的现代做法

- **元学习器**：S-learner（单模型带 T 特征）、T-learner（分模型）；
- **双重稳健（DR learner）**：倾向 + 结果双模型，其中一个对即无偏；
- **因果森林**：异质性无偏分割 + 诚实估计；
- 落地：uplift 模型决定「给谁发券/推哪个内容」——从 ATE 到决策的最后一公里。

In [19]:
# DR 学习器（ATE 版）：倾向 + 结果双模型
def dr_ate(T, y, ps, X):
    b, *_ = np.linalg.lstsq(np.column_stack([np.ones(len(y)), X, T]), y, rcond=None)
    m1 = np.column_stack([np.ones(len(y)), X, np.ones(len(y))]) @ b
    m0 = np.column_stack([np.ones(len(y)), X, np.zeros(len(y))]) @ b
    ps = np.clip(ps, 0.01, 0.99)
    return np.mean(m1 - m0 + T*(y-m1)/ps - (1-T)*(y-m0)/(1-ps))

rng = np.random.RandomState(17)
n = 1000
X = rng.normal(0, 1, n)
ps = 1/(1+np.exp(-0.6*X))
T = rng.rand(n) < ps
y = 2.0*T + 1.0*X + rng.normal(0, 1, n)
est = dr_ate(T, y, ps, X[:, None])
print('DR 估计 = %.3f（真值 2.0）| 朴素 = %.3f' % (est, y[T == 1].mean()-y[T == 0].mean()))
assert abs(est-2.0) < 0.15
print('✓ 双重稳健：任一模型正确即可无偏（比单一 IPW 稳）')

DR 估计 = 2.118（真值 2.0）| 朴素 = 2.710
✓ 双重稳健：任一模型正确即可无偏（比单一 IPW 稳）


## ⚠️ 因果推断的常见失败模式

1. 混淆未测且不可代理（只能敏感性分析）；
2. 把「控制变量越多越好」——控制中介/碰撞器反而开偏差；
3. PSM 无重叠仍强行匹配（外推虚构）；
4. DID 平行趋势不成立还硬跑（要事件研究图）；
5. IV 弱工具/排他性可疑照用；
6. 把相关/可预测当因果汇报（缺 DAG 与识别策略）。

## 🎤 因果三面追问（10 连）

Q「后门集合怎么找？」→ 阻断所有后门路径且不经过碰撞器（可画图推导）；
Q「中介能控制吗？」→ 不能当混淆控——它会把直接效应也吃掉；
Q「弱工具的补救？」→ 找更强 Z、多个 Z 检验、或放弃 IV 换设计；
Q「SCM 能做反事实？」→ 能：do-演算/结构方程推反事实分布；
Q「uplift 模型验证？」→ 留存测试集算真实增量（模拟/真实实验）
Q「因果森林 vs 随机森林？」→ 同形不同目标：分割准则与诚实估计保证无偏 CATE；
Q「DAG 画错怎么办？」→ 敏感性分析 + 替代 DAG 稳定性检验（方向一致才可信）；
Q「观察数据能得 ATE？」→ 能但依赖假设；报「在假设成立下估计值」；
Q「安慰剂检验怎么做？」→ 提前干预/无关结局重跑——应无效；
Q「给老板讲因果？」→ 一张 DAG + 一个调整估计 + 敏感性结论，别背公式。

In [20]:
# DAG 绘制：混淆 vs 碰撞器（因果结构第一课）
def draw_dag(edges, pos, title):
    plt.figure(figsize=(4.2, 3.0))
    for a, b in edges:
        plt.annotate('', xy=pos[b], xytext=pos[a],
                     arrowprops=dict(arrowstyle='->', color='k', lw=1.2))
    for k, (x, y) in pos.items():
        plt.scatter(x, y, s=700, c='white', edgecolors='k', zorder=3)
        plt.text(x, y, k, ha='center', va='center', fontsize=12)
    plt.title(title, fontsize=10); plt.axis('off'); plt.tight_layout(); plt.show()

draw_dag([('Z', 'T'), ('Z', 'Y'), ('T', 'Y')],
         {'Z': (0, 1), 'T': (1, 0), 'Y': (2, 1)}, '混淆：Z 打开 T→Y 的后门路径')
draw_dag([('T', 'C'), ('Y', 'C'), ('T', 'Y')],
         {'T': (0, 1), 'C': (1, 2), 'Y': (2, 1)}, '碰撞器：C 是 T 与 Y 的共同结果（不能控制）')

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\4117604400.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(title, fontsize=10); plt.axis('off'); plt.tight_layout(); plt.show()


In [21]:
# 混淆数值演示：朴素差异 ≠ 因果（控制 Z 后回归才对）
rng = np.random.RandomState(20)
n = 500
Z = rng.normal(0, 1, n)
T = (rng.rand(n) < 1/(1+np.exp(-Z))).astype(float)
Y = 0.5*T + 1.2*Z + rng.normal(0, 1, n)
naive = Y[T == 1].mean() - Y[T == 0].mean()
b, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), T, Z]), Y, rcond=None)
plt.scatter(T + 0.05*rng.randn(n), Y, s=6, alpha=0.3)
plt.xlabel('处理 T'); plt.ylabel('结果 Y')
plt.title('混淆下 T-Y 散点：肉眼看到的关系=相关，不是因果')
plt.tight_layout(); plt.show()
print('朴素差异 = %.2f | 控制 Z 后 = %.2f（真值 0.5）' % (naive, b[1]))

朴素差异 = 1.57 | 控制 Z 后 = 0.50（真值 0.5）


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\3631594892.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [22]:
# 倾向得分分布：重叠程度决定匹配/加权可行性
rng = np.random.RandomState(21)
n = 400
Z = rng.normal(0, 1, n)
ps = 1/(1+np.exp(-0.8*Z))
T = rng.rand(n) < ps
plt.hist(ps[T == 1], bins=30, density=True, alpha=0.5, label='处理组')
plt.hist(ps[T == 0], bins=30, density=True, alpha=0.5, label='对照组')
plt.legend(fontsize=8); plt.xlabel('倾向得分')
plt.title('PS 重叠：两端无支撑区 = 外推，匹配不可靠')
plt.tight_layout(); plt.show()
print('重叠区占比 = %.0f%%（建议 >80%%）' % (100*np.mean((ps > 0.1) & (ps < 0.9))))

重叠区占比 = 99%（建议 >80%）


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\521835550.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [23]:
# PSM 匹配前后平衡图：|SMD|<0.1 视为平衡
cov = ['年龄', '收入', '活跃度', '设备', '城市等级']
before = [0.45, 0.32, 0.28, 0.20, 0.12]
after = [0.06, 0.04, 0.05, 0.07, 0.03]
y = np.arange(5)
plt.plot(before, y, 'o-', label='匹配前')
plt.plot(after, y, 's-', label='匹配后')
plt.axvline(0.1, color='r', ls='--', label='|SMD|=0.1 阈值')
plt.yticks(y, cov); plt.legend(fontsize=8)
plt.title('平衡诊断：匹配后全部协变量 SMD<0.1（基线可比）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\3058993578.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [24]:
# DID 平行趋势：干预前两组走势须平行（前提可视化）
periods = ['T-2', 'T-1', 'T0(干预)', 'T+1', 'T+2']
ctrl = [50, 55, 62, 68, 75]
treat = [52, 57, 75, 82, 90]
plt.plot(range(5), ctrl, 'o-', label='对照组')
plt.plot(range(5), treat, 's-', label='处理组')
plt.axvline(2, color='k', ls='--', label='干预时刻')
plt.xticks(range(5), periods); plt.legend(fontsize=8)
plt.title('平行趋势：T0 之前斜率一致 → 之后分岔才算“处理效应”')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\1313408578.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [25]:
# DID 效应图解：处理组 − 对照组的变化差
c0, c1, t0, t1 = 55, 75, 57, 90
plt.plot([0, 1], [c0, c1], 'o-', label='对照组')
plt.plot([0, 1], [t0, t1], 's-', label='处理组')
plt.plot([0, 1], [t0, t0+(c1-c0)], 'r--', lw=2, label='反事实(无干预)')
plt.xticks([0, 1], ['T0(前)', 'T1(后)']); plt.legend(fontsize=8)
plt.title('DID = (90-57)-(75-55) = %d（剔除共同趋势的增量）' % ((t1-t0)-(c1-c0)))
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\845765862.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [26]:
# RDD 断点回归图：断点处跳跃 = 局部因果效应
rng = np.random.RandomState(22)
run = rng.uniform(-2, 2, 300)
treat = run >= 0
y = 1.5*treat + 0.8*run + rng.normal(0, 0.8, 300)
plt.scatter(run, y, s=6, alpha=0.3)
for side in (run < 0, run >= 0):
    xx = np.sort(run[side])
    plt.plot(xx, np.polyval(np.polyfit(run[side], y[side], 1), xx), 'r-', lw=2)
plt.axvline(0, color='k', ls='--', label='断点(录取线)')
plt.xlabel('运行变量'); plt.ylabel('结果'); plt.legend(fontsize=8)
plt.title('RDD：断点处跳跃≈局部因果（带宽内线性近似）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\3662667447.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [27]:
# uplift 累计增益曲线：排序后瞄准高增量用户
rng = np.random.RandomState(23)
n = 1000
gain = np.where(rng.rand(n) > 0.7, 1.2, 0.5)
cum = np.cumsum(np.sort(gain)[::-1])/np.sum(gain)
plt.plot(np.arange(1, n+1)/n, cum, lw=2)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('按增量排序后触达比例'); plt.ylabel('累计增益占比')
plt.title('uplift 曲线：前 30%% 用户贡献 >40%% 增益（定位施策对象）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\3229420355.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [28]:
# 2SLS 两阶段可视化：用 Z 预测的 T 去拟合 Y
rng = np.random.RandomState(24)
n = 500
Z = rng.normal(0, 1, n)
T = 0.6*Z + 0.5*rng.normal(0, 1, n)
Y = 1.0*T + 0.8*rng.normal(0, 1, n)
b1, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), Z]), T, rcond=None)
T_hat = np.column_stack([np.ones(n), Z]) @ b1
b2, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), T_hat]), Y, rcond=None)
plt.scatter(T, Y, s=6, alpha=0.3)
plt.plot(np.sort(T_hat), b2[0]+b2[1]*np.sort(T_hat), 'r-', lw=2, label='2SLS 拟合线')
plt.xlabel('T'); plt.ylabel('Y'); plt.legend(fontsize=8)
plt.title('2SLS：只用“由 Z 解释的干净部分”识别效应')
plt.tight_layout(); plt.show()
print('2SLS = %.3f | 朴素 OLS = %.3f（真值 1.0）' % (b2[1], np.cov(T, Y)[0, 1]/np.var(T)))

2SLS = 0.996 | 朴素 OLS = 1.004（真值 1.0）


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\1646184184.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [29]:
# 后门调整前后效应对比
ests = {'朴素(未调整)': 0.91, '调整 Z 后': 0.52, '真值': 0.50}
plt.bar(list(ests), list(ests.values()), color=['C1', 'C0', 'k'])
for k, v in ests.items(): plt.text(list(ests).index(k), v+0.02, '%.2f' % v, ha='center')
plt.title('后门调整：把混淆按住 → 估计逼近真值')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\799059813.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [30]:
# 中介路径系数图：总效应 = 直接 + 间接
plt.figure(figsize=(5.5, 3.0))
def arrow(a, b, lab, off=0.0):
    plt.annotate('', xy=b, xytext=a, arrowprops=dict(arrowstyle='->', lw=1.5))
    plt.text((a[0]+b[0])/2+off, (a[1]+b[1])/2+0.05, lab, ha='center', fontsize=10)
for k, p in [('T', (0, 1)), ('M', (1, 1.8)), ('Y', (2, 1))]:
    plt.scatter(*p, s=800, c='w', edgecolors='k', zorder=3)
    plt.text(p[0], p[1], k, ha='center', va='center', fontsize=13)
arrow((0, 1), (1, 1.8), 'a=1.0')
arrow((1, 1.8), (2, 1), 'b=1.2')
arrow((0, 1), (2, 1), "c'=0.8", off=0.12)
plt.title('中介模型：总效应 c = c′ + a·b = 2.0'); plt.axis('off')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\3674518322.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [31]:
# E-value 敏感性曲线：观测 RR 需要多强的未测混淆才能推翻
rr = np.linspace(1.2, 3, 19)
ev = rr + np.sqrt(rr*(rr-1))
plt.plot(rr, ev, lw=2, label='E-value')
plt.plot(rr, rr, 'k--', label='对角参考')
plt.legend(fontsize=8); plt.xlabel('观测 RR'); plt.ylabel('需要的未测混淆 RR')
plt.title('E-value：观测 RR=2 → 未测混淆至少 RR=2.41 才能解释')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\703542757.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [32]:
# 选择性流失 → 幸存者偏差（ATT 估计被污染）
rng = np.random.RandomState(25)
n = 400
T = rng.rand(n) < 0.5
y = rng.normal(5, 1, n) + 1.0*T
att = rng.rand(n) < (0.3 + 0.3*T)      # 处理组流失更多
obs_T, obs_y = T[~att], y[~att]
plt.scatter(T, y, s=6, alpha=0.4, label='完整')
plt.scatter(T[att], y[att], s=14, c='r', marker='x', label='流失(不可观测)')
plt.legend(fontsize=8); plt.title('选择性流失：红色不可观测 → 幸存者均值≠因果')
plt.tight_layout(); plt.show()
print('真实效应 = 1.00 | 幸存者估计 = %.2f（流失机制引入偏差）'
      % (obs_y[obs_T == 1].mean()-obs_y[obs_T == 0].mean()))

真实效应 = 1.00 | 幸存者估计 = 0.89（流失机制引入偏差）


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\2036891371.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [33]:
# 因果方法选择流程图：数据长什么样 → 用哪套工具
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.add_patch(plt.Rectangle((0.40, 0.66), 0.2, 0.24, fc='#ffe9d9', ec='#b45309', lw=1.2))
ax.text(0.50, 0.78, '能做随机实验？', ha='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.72, 0.35), 0.24, 0.32, fc='#e2efda', ec='#376027', lw=1.1))
ax.text(0.84, 0.51, 'RCT / AB', ha='center', fontsize=8)
ax.annotate('', xy=(0.72, 0.51), xytext=(0.60, 0.68), arrowprops=dict(arrowstyle='->', lw=1.1))
ax.text(0.66, 0.60, '是', fontsize=9, color='#376027')
ax.add_patch(plt.Rectangle((0.04, 0.30), 0.26, 0.36, fc='#dbe9ff', ec='#1f4e79', lw=1.1))
ax.text(0.17, 0.44, '观测数据\n(处理非随机)', ha='center', fontsize=8)
ax.annotate('', xy=(0.40, 0.42), xytext=(0.305, 0.46), arrowprops=dict(arrowstyle='->', lw=1.1))
ax.text(0.33, 0.50, '否', fontsize=9, color='#b45309')
ax.text(0.17, 0.10, '倾向分可建模→\nPSM/IPW/重加权', ha='center', fontsize=8, color='#1f4e79')
ax.text(0.42, 0.10, '有前后两期→\nDID（平行趋势）', ha='center', fontsize=8, color='#1f4e79')
ax.text(0.68, 0.10, '有外生工具→\n2SLS/IV', ha='center', fontsize=8, color='#1f4e79')
ax.text(0.93, 0.10, '有 DAG→\n后门调整', ha='center', fontsize=8, color='#1f4e79')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('因果方法选型：数据特征(可实验/可建模/时间/工具/图) 决定方法论', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 面试话术：先讲能做什么实验，再讲观测数据的假设与代价')

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\3264526631.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


✓ 面试话术：先讲能做什么实验，再讲观测数据的假设与代价


## 🧮 因果推断核心公式（默写版）

**反事实/ATE**：$\tau=E[Y(1)-Y(0)]$；观测数据有 $E[Y|T=1]-E[Y|T=0]\ne\tau$（混淆）。

**倾向得分**：$e(x)=P(T=1|X=x)$；
**IPW**：$\hat\tau_{IPW}=\dfrac{1}{n}\sum\left(\dfrac{T_i Y_i}{e_i}-\dfrac{(1-T_i)Y_i}{1-e_i}\right)$。

**DID**：$\hat\tau_{DID}=(\bar Y_{T,2}-\bar Y_{T,1})-(\bar Y_{C,2}-\bar Y_{C,1})$；
回归式 $Y=\beta_0+\beta_1 T+\beta_2 After+\tau\,T\!\cdot\!After$。

**IV/2SLS**：第一阶段 $T=\gamma Z+\eta$，第二阶段 $Y=\tau\hat T+\varepsilon$；
$\hat\tau_{IV}=\dfrac{\mathrm{Cov}(Z,Y)/\mathrm{Var}(Z)}{\mathrm{Cov}(Z,T)/\mathrm{Var}(Z)}$。

**后门调整**：$P(y|do(t))=\sum_c P(y|t,c)P(c)$（$C$ 阻断所有后门路径）。

**中介**：总效应 $c=c'+ab$。

> 一句话每方法：PSM/IPW=按选择概率重加权；DID=双重差分消趋势；IV=用外生变动代替处理。

In [34]:
# IPW 数字示例：混淆存在时，加权恢复 ATE
rng = np.random.RandomState(1)
n = 20000
x1 = rng.randn(n)
ps = 1/(1+np.exp(-(0.5 + 0.8*x1)))
t = (rng.rand(n) < ps).astype(float)
y0 = 0.5*x1 + rng.randn(n)
y1 = y0 + 1.5
y = y1*t + y0*(1-t)
naive = y[t == 1].mean() - y[t == 0].mean()
ipw = np.mean(t*y/ps) - np.mean((1-t)*y/(1-ps))
print('朴素差异 = %.3f（x1 混淆 → 偏离 1.5）| IPW = %.3f | 真值 = 1.5' % (naive, ipw))
assert abs(ipw - 1.5) < 0.08
print('✓ IPW 把每个样本按 1/倾向得分 加权 → 处理组/对照组在 X 上重新“均衡”')

朴素差异 = 1.834（x1 混淆 → 偏离 1.5）| IPW = 1.485 | 真值 = 1.5
✓ IPW 把每个样本按 1/倾向得分 加权 → 处理组/对照组在 X 上重新“均衡”


## 📜 因果分析执行伪代码

```
1. 画 DAG：列出所有变量与路径（先验知识）
2. 识别：找混淆集（阻断后门）→ 选方法（匹配/IPW/DID/IV）
3. 估计：
   PSM：logit(T~X) → 近邻/核匹配 → 配对后差分
   IPW：e=logit 预测；τ=mean(T·Y/e) - mean((1-T)·Y/(1-e))
   DID：两期两组的组间差再相减
4. 稳健性：重叠性检查、平衡性检查（SMD<0.1）、安慰剂检验
5. 报告：效应 + CI + 假设与局限
```

> 加餐：敏感性分析——未观测混淆要多大才能推翻结论（E-value）。

## 🎤 因果推断追问（8 连·第二轮）

Q「倾向得分匹配什么时候不可用？」→ 重叠性差（两组 ps 无交集）、样本少、高维易过拟合；
Q「DID 的核心假设？」→ 平行趋势（无处理时两组同趋势）→ 用多期画趋势检验；
Q「工具变量怎么找？」→ 外生且只经处理影响结果（如距离、天气、政策时点）；
Q「后门与前门区别？」→ 后门=消除混淆；前门=经中介识别（无需混淆可观测）；
Q「为什么相关不等于因果？」→ 混淆/反向/选择偏差三座大山；
Q「CATE 与 ATE 区别？」→ CATE 分群异质性（谁受益多）→ 个性化决策；
Q「自然实验是什么？」→ 外生冲击(政策/事故/平台故障)天然分组 → 近似随机；
Q「因果森林/元学习怎么做？」→ 双机器学习/DR 估计 + 树集成 → 估计 CATE。

## ✅ 自测（10 题）

1. ATE 定义？（E[Y(1)-Y(0)]）
2. 混淆变量是什么？（同时影响 T 与 Y 的变量）
3. IPW 的权重是什么？（1/倾向得分）
4. DID 消除什么？（组别固定差异 + 共同时间趋势）
5. IV 三条件？（相关性/外生性/排他性）
6. 后门路径怎么阻断？（按混淆集调整 C）
7. 中介效应怎么分解？（总=直接+间接 a·b）
8. 匹配的好处？（可比子样本上做差分，直观）
9. 平行趋势如何检验？（多期数据画走势/事件研究）
10. 观测研究能证明因果吗？（不能证，只能逼近；假设+稳健性分析）

In [35]:
# DID 手算示例：2 期 2 组（消除时间趋势后的真实效应）
g_treat = [10.0, 15.0]; g_ctrl = [12.0, 14.0]
d_treat = g_treat[1] - g_treat[0]
d_ctrl = g_ctrl[1] - g_ctrl[0]
did = d_treat - d_ctrl
print('处理组变化 = %.1f | 对照组变化 = %.1f（时间趋势+其他冲击）' % (d_treat, d_ctrl))
print('DID = %.1f - %.1f = %.1f（对照组净化掉公共趋势后的处理效应）' % (d_treat, d_ctrl, did))
assert abs(did - 3.0) < 1e-9
print('✓ 验证：若对照组效应=0，处理组 5.0 就直接是效应——但现实总有趋势 → 必须 DID')

处理组变化 = 5.0 | 对照组变化 = 2.0（时间趋势+其他冲击）
DID = 5.0 - 2.0 = 3.0（对照组净化掉公共趋势后的处理效应）
✓ 验证：若对照组效应=0，处理组 5.0 就直接是效应——但现实总有趋势 → 必须 DID


In [36]:
# 后门路径与调整示意：混淆 C 同时指向 T 与 Y
fig, ax = plt.subplots(figsize=(7.2, 2.8))
def nbox(lab, x, y):
    ax.add_patch(plt.Rectangle((x-0.08, y-0.09), 0.16, 0.18, fc='#dbe9ff', ec='#1f4e79', lw=1.2, transform=ax.transAxes))
    ax.text(x, y, lab, ha='center', va='center', fontsize=10, transform=ax.transAxes)
nbox('C', 0.5, 0.72); nbox('T', 0.18, 0.35); nbox('Y', 0.82, 0.35)
ax.annotate('', xy=(0.42, 0.60), xytext=(0.26, 0.42), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.58, 0.60), xytext=(0.74, 0.42), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.74, 0.35), xytext=(0.26, 0.35), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.text(0.5, 0.86, '混淆路径 T←C→Y（后门）', ha='center', fontsize=10, color='#b45309', transform=ax.transAxes)
ax.text(0.5, 0.10, '调整：E[Y|do(T=t)] = Σ_c P(Y|T=t,C=c)·P(C=c)', ha='center', fontsize=10, color='#1f4e79', transform=ax.transAxes)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('后门调整：按混淆变量分层平均（或回归/加权实现同一目标）', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 不调整 → T 的系数里混入 C 的效应；调整后 = 纯净的处理效应')

C:\Users\24260\AppData\Local\Temp\ipykernel_45692\1669477261.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


✓ 不调整 → T 的系数里混入 C 的效应；调整后 = 纯净的处理效应


In [37]:
# 工具变量数字示例：2SLS 手算（混淆存在时 IV 仍无偏）
rng = np.random.RandomState(4)
n = 5000
z = rng.randn(n)
u = rng.randn(n)
t = 0.8*z + 0.5*u + rng.randn(n)
y = 2.0*t + 1.0*u + rng.randn(n)
ols = np.polyfit(t, y, 1)[0]
b, a = np.polyfit(z, t, 1)
t_hat = b*z + a
iv = np.polyfit(t_hat, y, 1)[0]
print('OLS = %.3f（u 混淆 → 偏大）| 2SLS = %.3f | 真值 = 2.0' % (ols, iv))
assert abs(iv - 2.0) < 0.15
print('✓ Z 只“推动”T（第一步），不碰 u → 第二阶段得到的斜率 = 因果效应')

OLS = 2.256（u 混淆 → 偏大）| 2SLS = 1.989 | 真值 = 2.0
✓ Z 只“推动”T（第一步），不碰 u → 第二阶段得到的斜率 = 因果效应


In [38]:
# 中介效应路径图：X → M → Y
fig, ax = plt.subplots(figsize=(7.2, 2.7))
def nbox(lab, x, y):
    ax.add_patch(plt.Rectangle((x-0.08, y-0.09), 0.16, 0.18, fc='#dbe9ff', ec='#1f4e79', lw=1.2, transform=ax.transAxes))
    ax.text(x, y, lab, ha='center', va='center', fontsize=11, transform=ax.transAxes)
nbox('X', 0.15, 0.5); nbox('M', 0.5, 0.72); nbox('Y', 0.85, 0.5)
ax.annotate('', xy=(0.42, 0.66), xytext=(0.23, 0.54), arrowprops=dict(arrowstyle='->', lw=1.4))
ax.text(0.31, 0.64, 'a', fontsize=12, color='#b45309', transform=ax.transAxes)
ax.annotate('', xy=(0.77, 0.62), xytext=(0.58, 0.70), arrowprops=dict(arrowstyle='->', lw=1.4))
ax.text(0.70, 0.68, 'b', fontsize=12, color='#b45309', transform=ax.transAxes)
ax.annotate('', xy=(0.77, 0.5), xytext=(0.23, 0.5), arrowprops=dict(arrowstyle='->', lw=1.4))
ax.text(0.5, 0.44, "c'（直接效应）", ha='center', fontsize=10, color='#1f4e79', transform=ax.transAxes)
ax.text(0.5, 0.10, "总效应 c = c' + a·b | 中介占比 = a·b / c", ha='center', fontsize=10, color='#555', transform=ax.transAxes)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('中介分析：处理如何“经由”M 起作用（如：价格→满意度→复购）', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 中介≠混淆：M 在 T 之后（机制），C 在 T 之前（混淆）——时序是判据')

✓ 中介≠混淆：M 在 T 之后（机制），C 在 T 之前（混淆）——时序是判据


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\2441706010.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🍵 白话开篇：相关 ≠ 因果，中间隔着三座大山

“冰淇淋销量 ↑ 和溺水人数 ↑ 强相关”——谁也不会把冰淇淋当溺水原因：因为**温度**在背后。
观测数据里，相关可能来自三种非因果机制（背下来）：

1. **混淆（confounding）**：共同原因 C 同时推着 T 和 Y（温度→冰淇淋&游泳）——
   C 不控制，T 的“表观效应”就混进了 C 的贡献；
2. **反向因果**：其实是 Y 影响 T（更瘦的人更常健身——是“健康→健身”还是“健身→健康”？）；
3. **选择偏差/样本偏倚**：数据里 T 不是随机来的（看到“买会员的人消费高”——
   有钱人本来就爱买会员爱花钱）。

**因果要回答的反事实问题**：同一批人，**既**接受处理（买会员）**又**不接受，
两个世界的消费差多少？——现实中只能看到一个世界 → 必须用设计/假设补出另一个。

**一句话**：AB 用“随机化”直接造出反事实；观测数据要**靠假设+统计方法**来补。

## 🍵 白话讲透倾向得分：把“选择”变成“概率”，再按概率平衡

**问题**：观测数据里“买会员”不是随机——有钱人、活跃的人更爱买。
直接比“买 vs 不买”的消费，分不清是“会员的效果”还是“本来就更有钱”。

**倾向得分 e(x) = P(T=1 | X=x)**：用协变量（收入、活跃度…）预测“这个人买会员的概率”。

**为什么平衡倾向得分就够了（关键直觉）**：
- 两个 e(x) 相同的人，在**所有**影响选择的 X 上分布一样（可观测世界等价）；
- 于是“一个买、一个不买”≈ 随机对照——可比较！

**三种用法（面试三连问）**：
- **匹配（PSM）**：为每个处理组找 e 最接近的对照组（r10 代码示例：SMD 前后对比）；
- **加权（IPW）**：给样本按 1/e 或 1/(1−e) 加权，把两组“拉成”与总体同分布；
- **分层**：按 e 分 5 层，层内比较再合并。

**一句话**：PS = 用可观测变量“模拟随机化”——好坏取决于 X 是否抓住所有混淆（不可观测混淆是硬伤）。

## 🍵 白话讲透中介：处理是怎么“拐弯”起作用的

促销 T → 销量 Y，路径不止一条：
- **直接**：促销本身让人想买；
- **间接（经由 M）**：促销 → 更多人进店逛（M）→ 顺带买了别的东西。

**中介分析回答**：“总效果里，有多少是经由 M 实现的？”

**经典三回归（Baron-Kenny，背下来）**：
1. $Y=\beta_0+cT$（总效应 c）；
2. $M=\alpha_0+aT$（处理对中介的效应 a）；
3. $Y=\gamma_0+c'T+bM$（控制 M 后处理效应 c' + M 的效应 b）。

**符号解释**：c = 总效应；c' = 直接效应；**a·b = 间接效应**；
中介占比 = a·b/c。完全中介：c'≈0；部分中介：c' 仍显著。

**为什么“时序是判据”**：M 必须发生在 T **之后**、Y **之前**（T→M→Y 的顺序）——
把混淆 C（T 之前）放进 M 就错了。

**一句话**：中介 = 拆解“怎么起作用”；混淆 = 弄清“是不是它起作用”——别混。

## 🔍 公式拆解 ①：IPW 加权——把两组“补”成随机

$$\hat\tau_{IPW}=\frac1n\sum_{i}\frac{T_iY_i}{e_i}-\frac1n\sum_{i}\frac{(1-T_i)Y_i}{1-e_i}$$

**逐符号解释**：
- $T_i$：第 i 人是否处理（0/1）；$Y_i$：结果；$e_i=P(T_i=1|X_i)$：倾向得分；
- 第一项：**处理组**每人权重 $1/e_i$——买会员概率低的（穷人）权重放大，代表“全体中不常买的人”；
- 第二项：**对照组**每人权重 $1/(1-e_i)$——补“本来会买但没买的人”；
- 两项均值之差 = 全体平均处理效应（ATE）。

**为什么能去偏（直觉）**：随机实验里两组协变量分布天然一样；
观测数据里不一样 → 用权重把“少的那个群体”放大、把“多的那个群体”缩小，
**硬把两组拉成与总体同分布** → 之后差异只来自处理。

**注意**：
- 极端权重（e 接近 0/1）方差爆炸 → 截断（trim）或权重归一；
- **重叠假设**：每人都要有正概率进任一组（e 不在 0/1 附近），否则加权无意义；
- e 模型错 → 有偏（双稳健 DR 估计可补救）。

**一句话**：IPW = 给样本发“代表票”，票面 = 1/被选择的概率。

## 🔍 公式拆解 ②：DID——两次差分，消掉共同趋势

$$\hat\tau_{DID}=\big(\bar Y_{T,after}-\bar Y_{T,before}\big)-\big(\bar Y_{C,after}-\bar Y_{C,before}\big)$$

**逐符号解释**：
- $\bar Y_{T,after}$：处理组改造后的均值；$\bar Y_{C,before}$：对照组改造前的均值；
- 第一差（处理组前后差）：包含“政策效果 + 共同趋势 + 处理组特有漂移”；
- 第二差（对照组前后差）：**只有共同趋势**（对照组没被处理）；
- 两差相减：共同趋势抵消，剩下 = 政策效果。

**关键假设：平行趋势**——没有政策时，两组均值差**保持恒定**（走势平行）。
验证方法：看政策**前**多期两条线的斜率是否一致（平行趋势检验图）。

**回归等价式**：$Y=\beta_0+\beta_1T+\beta_2\text{After}+\beta_3(T\times\text{After})+\varepsilon$，
$\beta_3$ 就是 DID 效应（本书 r9 数字示例验证 = 3.0）。

**一句话**：DID = “对照组的变化 = 我组没有政策时本会有的变化”，减掉它。

## 🔍 公式拆解 ③：工具变量（2SLS）——找个“只推 T、不碰 Y”的推手

$$\hat\beta_{IV}=\frac{\text{Cov}(Z,Y)}{\text{Cov}(Z,T)}\quad(\text{二阶段最小二乘：先 T 对 Z 回归取拟合值，再 Y 对拟合值回归})$$

**逐符号解释**：
- $Z$：工具变量（如“距门店距离”：只影响是否到店消费 T，不影响消费额 Y 的其他决定）；
- $\text{Cov}(Z,T)$：Z 与 T 的相关（**相关性**）；$\text{Cov}(Z,Y)$：Z 与 Y 的相关；
- 比值 = “Z 每推一点 T，Y 跟着变多少”——把 T 的因果效应从“Z→T→Y”这条干净链上量出来。

**两个假设（面试必背）**：
1. **相关性**：Cov(Z,T)≠0（Z 确实推 T）；
2. **排他性**：Z 只通过 T 影响 Y（Z 与误差项无关）——不可直接检验，靠论证。

**直觉（比喻）**：想知道“喝酒对健康”的因果，直接回归会被“爱喝酒的人也爱熬夜”混淆；
用“酒价（Z）”当推手：酒价只影响喝酒量、不影响其他生活方式 →
酒价引起的喝酒量变化对健康的影响 = 干净的因果。

**弱工具警告**：Cov(Z,T) 太小 → 估计方差巨大、还偏（弱 IV 病）。

**一句话**：IV = 找一个“实验般的外来推手”，把处理变量里“干净”的那部分变化挑出来用。

## 🔍 公式拆解 ④：后门调整公式

$$P(y\mid do(t))=\sum_{c}P(y\mid t,c)P(c)$$

**逐符号解释**：
- $do(t)$：**干预符号**——“强制把 T 设为 t”（不同于“观察到 t”）；
- $P(y|t,c)$：**层内**（控制住混淆 c）T 对 Y 的条件分布；
- $P(c)$：混淆变量的总体分布；
- 求和：**层内估计 × 总体占比**加权合并。

**为什么“层内 + 加权”**：
- 直接看 $P(y|t)$：混进了 c 的影响（c 同时推 t 和 y）；
- 先固定 c（层内比较，c 不再捣乱），再按总体占比加权——
  还原“如果所有人都经历 do(t)”的世界；
- 这就是“控制混杂变量”的数学形式。

**适用条件**：所有混淆变量都被观测到（无未观测混杂）——
多变量时先按因果图找**最小充分调整集**（别过度控制中介！）。

**一句话**：后门调整 = “分层比一比，再按人口加权” = 把观察数据伪装成随机实验。

In [39]:
# 后门调整手算：性别 C 混杂“买会员 T”与“消费 Y”，比较朴素差与调整差
rng = np.random.RandomState(4)
n = 4000
c = rng.binomial(1, 0.5, n)                  # 性别：0 男 / 1 女
t = rng.binomial(1, 0.3 + 0.35*c)             # 女性更爱买会员（选择偏差）
eff = 2.0
y = 3.0 + eff*t + 1.5*c + rng.randn(n)        # 真 ATE = 2.0；c 直接推 y（混淆）
ate_naive = y[t == 1].mean() - y[t == 0].mean()
adj = 0.0
for cc in [0, 1]:
    pc = (c == cc).mean()
    e1 = y[(t == 1) & (c == cc)].mean()
    e0 = y[(t == 0) & (c == cc)].mean()
    adj += pc*(e1 - e0)                        # Σ P(c)·[层内差]
print('朴素比较 ATE = %.3f（偏高：性别混在里头）' % ate_naive)
print('后门调整 ATE = %.3f | 真值 = %.1f' % (adj, eff))
assert abs(adj - eff) < 0.15 and abs(ate_naive - eff) > abs(adj - eff)
print('✓ 层内先差分（同性别人群才可比）→ 按性别占比加权 = 公式 Σ P(c)P(y|t,c)')

朴素比较 ATE = 2.497（偏高：性别混在里头）
后门调整 ATE = 1.984 | 真值 = 2.0
✓ 层内先差分（同性别人群才可比）→ 按性别占比加权 = 公式 Σ P(c)P(y|t,c)


In [40]:
# 倾向得分匹配：匹配前后协变量平衡（SMD）对比
rng = np.random.RandomState(3)
n = 500
x1 = rng.randn(n); x2 = rng.randn(n)
ps = 1/(1+np.exp(-(-0.5 + 0.8*x1 + 0.6*x2)))   # 选择机制（真实生成）
t = (rng.rand(n) < ps).astype(float)
y = 2.0*t + 0.7*x1 + 0.5*x2 + rng.randn(n)
def smd(v, a, b):
    return (v[a].mean()-v[b].mean())/np.sqrt((v[a].std()**2+v[b].std()**2)/2)
lp = np.log(ps/(1-ps))
ctl = np.where(t == 0)[0]
tm = np.where(t == 1)[0]
match = np.array([ctl[np.argmin(np.abs(lp[ctl] - lp[i]))] for i in tm])  # 1:1 最近邻
smd_before = np.array([smd(x1, tm, ctl), smd(x2, tm, ctl)])
smd_after = np.array([smd(x1, tm, match), smd(x2, tm, match)])
print('SMD 匹配前: x1=%.3f x2=%.3f | 匹配后: x1=%.3f x2=%.3f（经验线 |SMD|<0.1）'
      % (smd_before[0], smd_before[1], smd_after[0], smd_after[1]))
plt.figure(figsize=(7, 2.6))
xx = np.arange(2)
plt.bar(xx-0.18, np.abs(smd_before), 0.36, label='匹配前', color='#b45309')
plt.bar(xx+0.18, np.abs(smd_after), 0.36, label='匹配后', color='#1f4e79')
plt.axhline(0.1, color='k', ls='--', lw=1, label='|SMD|=0.1 经验线')
plt.xticks(xx, ['x1', 'x2']); plt.ylabel('|SMD|')
plt.legend(fontsize=8); plt.title('倾向得分匹配：把两组的协变量“拉平”', fontsize=10)
plt.tight_layout(); plt.show()
assert np.abs(smd_after).max() < np.abs(smd_before).max()
print('✓ 匹配后协变量平衡 → 两组可比 → 再比 Y 才接近因果（SMD 是报告标配）')

SMD 匹配前: x1=0.676 x2=0.487 | 匹配后: x1=0.010 x2=0.006（经验线 |SMD|<0.1）


✓ 匹配后协变量平衡 → 两组可比 → 再比 Y 才接近因果（SMD 是报告标配）


C:\Users\24260\AppData\Local\Temp\ipykernel_45692\1423072232.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⚠️ 本章高频踩坑

1. **未观测混杂**：所有方法都假设“该控的都控了”——有遗漏就全盘皆输（敏感性分析）；
2. **重叠不足**：e 接近 0/1 → IPW/匹配不可靠——先看倾向得分分布重叠；
3. **过度控制中介**：把 T→M→Y 里的 M 当混杂控掉 = 把间接效应也删了；
4. **平行趋势未验证**：DID 结论建立在“没政策也平行”上——必须画前趋势；
5. **弱工具**：IV 相关性弱 → 估计又偏又散；排他性无法检验，靠“故事”；
6. **幸存者/选择偏差**：样本不是随机来的（只看付费用户）→ 结论只对那部分人成立；
7. **PSM 被滥用**：匹配不等于随机化；不可观测混杂依旧；报告必须带平衡表；
8. **把因果当预测**：XGBoost 预测“谁买”≠ 因果效应“该不该打折”——预测要相关性、因果要干预。

## 🧭 因果方法全家桶：选型地图

| 方法 | 数据形态 | 核心假设 | 输出 |
|---|---|---|---|
| 随机实验 AB | 可随机化 | 随机化成立 | 最干净 ATE |
| IPW | 观测+倾向得分 | 无未观测混杂+重叠 | ATE |
| PSM | 观测 | 同上（匹配后） | ATT 为主 |
| 分层/回归 | 观测 | 正确控制 C | ATE（模型敏感） |
| DID | 两期两组面板 | 平行趋势 | 政策效应 |
| IV/2SLS | 有工具 Z | 相关+排他 | 局部平均效应 LATE |
| 断点 RDD | 有强制阈值 | 阈值附近连续 | 阈值局部效应 |
| 后门/前门 | 因果图 | 图假设+无未观测混杂 | 任意干预效应 |
| 双稳健 DR | 观测 | 倾向或结果模型其一正确 | ATE（稳健） |

**一句话选型**：能随机 → AB；有自然阈值 → RDD；有工具 → IV；有面板 → DID；
只有截面 → PS/IPW/后门；都会 → DR。

## 🏁 本章学完清单

- [ ] 能讲清三座大山（混淆/反向/选择）各是什么；
- [ ] 能解释倾向得分为什么能“模拟随机化”；
- [ ] 能手写 IPW 公式并逐符号解释（权重=1/e）；
- [ ] 能写出 DID 双重差分并解释平行趋势假设；
- [ ] 能说出 IV 的两个假设与弱工具危害；
- [ ] 能解释后门公式“分层+加权”的直觉；
- [ ] 能做：后门手算、PSM 平衡性检查（本书示例）；
- [ ] 能按数据形态选方法（选型地图）。

> 下一站：数据科学的最后一站——指标与八股（下一本 notebook）。